# Are the contact maps correct?

In [13]:
import pandas as pd

SUMMARY_FILE = "/nfs/scratch/pdb_dimers/contact_maps/contact_map_summary.tsv"
INTERACTION_FILE = "/nfs/scratch/pdb_dimers/final_filtered_interactions_with_partitions.tsv"
SEQUENCE_FILE = "/nfs/scratch/pdb_dimers/unique_sequences_with_partitioning.tsv"
ENTITY_SEQUENCE_PATH = "/nfs/scratch/pdb_dimers/entity_sequences.tsv"

In [15]:
summary_df = pd.read_csv(SUMMARY_FILE, sep="\t")
interaction_df = pd.read_csv(INTERACTION_FILE, sep="\t")

summary_df["assembly_id"] = summary_df["pdb_id"] + "-" + summary_df["assembly_number"].astype(str)

summary_df = summary_df.merge(interaction_df[["assembly_id", "new_cluster_pair"]], on="assembly_id", how="left")

summary_df.head()

,row_index,pdb_id,assembly_number,structure_file,output_file,requested_entity_name_1,requested_entity_name_2,requested_entity_1,requested_entity_2,remapped_entity_1,...,resolved_residues_1,resolved_residues_2,missing_residues_1,missing_residues_2,known_pairs,unknown_pairs,contacts,shape,assembly_id,new_cluster_pair
0,0,10BL,1,10bl-assembly1.cif.gz,10bl-assembly1.npy,10BL_1,10BL_1,1,1,1,...,338,326,23,35,110188,20133,139,361x361,10BL-1,"fix_18845_100,fix_18845_100"
1,1,10FT,1,10ft-assembly1.cif.gz,10ft-assembly1.npy,10FT_1,10FT_2,1,2,1,...,213,562,604,449,119706,706281,125,817x1011,10FT-1,"fix_14530_100,fix_37352_100"
2,2,10JU,1,10ju-assembly1.cif.gz,10ju-assembly1.npy,10JU_1,10JU_2,1,2,1,...,270,252,92,110,68040,63004,41,362x362,10JU-1,"fix_18848_100,fix_18846_100"
3,3,10LI,1,10li-assembly1.cif.gz,10li-assembly1.npy,10LI_1,10LI_1,1,1,1,...,462,462,16,16,213444,15040,262,478x478,10LI-1,"fix_31296_100,fix_31296_100"
4,4,10NM,1,10nm-assembly1.cif.gz,10nm-assembly1.npy,10NM_1,10NM_1,1,1,1,...,448,449,48,47,201152,44864,277,496x496,10NM-1,"fix_20451_100,fix_20451_100"


In [16]:
sequence_df = pd.read_csv(SEQUENCE_FILE, sep="\t")

summary_df[["cluster1", "cluster2"]] = summary_df["new_cluster_pair"].str.split(",", expand=True)

summary_df

,row_index,pdb_id,assembly_number,structure_file,output_file,requested_entity_name_1,requested_entity_name_2,requested_entity_1,requested_entity_2,remapped_entity_1,...,missing_residues_1,missing_residues_2,known_pairs,unknown_pairs,contacts,shape,assembly_id,new_cluster_pair,cluster1,cluster2
0,0,10BL,1,10bl-assembly1.cif.gz,10bl-assembly1.npy,10BL_1,10BL_1,1,1,1,...,23,35,110188,20133,139,361x361,10BL-1,"fix_18845_100,fix_18845_100",fix_18845_100,fix_18845_100
1,1,10FT,1,10ft-assembly1.cif.gz,10ft-assembly1.npy,10FT_1,10FT_2,1,2,1,...,604,449,119706,706281,125,817x1011,10FT-1,"fix_14530_100,fix_37352_100",fix_14530_100,fix_37352_100
2,2,10JU,1,10ju-assembly1.cif.gz,10ju-assembly1.npy,10JU_1,10JU_2,1,2,1,...,92,110,68040,63004,41,362x362,10JU-1,"fix_18848_100,fix_18846_100",fix_18848_100,fix_18846_100
3,3,10LI,1,10li-assembly1.cif.gz,10li-assembly1.npy,10LI_1,10LI_1,1,1,1,...,16,16,213444,15040,262,478x478,10LI-1,"fix_31296_100,fix_31296_100",fix_31296_100,fix_31296_100
4,4,10NM,1,10nm-assembly1.cif.gz,10nm-assembly1.npy,10NM_1,10NM_1,1,1,1,...,48,47,201152,44864,277,496x496,10NM-1,"fix_20451_100,fix_20451_100",fix_20451_100,fix_20451_100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22088,22088,9ZMZ,1,9zmz-assembly1.cif.gz,9zmz-assembly1.npy,9ZMZ_1,9ZMZ_1,1,1,1,...,13,13,9801,2743,203,112x112,9ZMZ-1,"fix_39175_100,fix_39175_100",fix_39175_100,fix_39175_100
22089,22089,9ZNN,1,9znn-assembly1.cif.gz,9znn-assembly1.npy,9ZNN_1,9ZNN_1,1,1,1,...,35,32,148608,26953,117,419x419,9ZNN-1,"fix_32925_100,fix_32925_100",fix_32925_100,fix_32925_100
22090,22090,9ZNO,1,9zno-assembly1.cif.gz,9zno-assembly1.npy,9ZNO_1,9ZNO_1,1,1,1,...,34,34,109561,23664,67,365x365,9ZNO-1,"fix_13069_100,fix_13069_100",fix_13069_100,fix_13069_100
22091,22091,9ZVM,1,9zvm-assembly1.cif.gz,9zvm-assembly1.npy,9ZVM_1,9ZVM_1,1,1,1,...,411,411,201601,537999,62,860x860,9ZVM-1,"fix_24677_100,fix_24677_100",fix_24677_100,fix_24677_100


In [17]:
sequence_lengths = sequence_df.set_index("new_cluster_id")["sequence"].str.len()

summary_df["new_shape1"] = summary_df["cluster1"].map(sequence_lengths)
summary_df["new_shape2"] = summary_df["cluster2"].map(sequence_lengths)

missing_lengths = summary_df[["new_shape1", "new_shape2"]].isna().any(axis=1)
if missing_lengths.any():
    display(summary_df.loc[missing_lengths, ["pdb_id", "cluster1", "cluster2", "new_shape1", "new_shape2"]])
    raise ValueError("Some cluster IDs were not found in sequence_df")

summary_df["new_shape1"] = summary_df["new_shape1"].astype(int)
summary_df["new_shape2"] = summary_df["new_shape2"].astype(int)
summary_df["control_shape"] = summary_df["new_shape1"].astype(str) + "x" + summary_df["new_shape2"].astype(str)

summary_df

,row_index,pdb_id,assembly_number,structure_file,output_file,requested_entity_name_1,requested_entity_name_2,requested_entity_1,requested_entity_2,remapped_entity_1,...,unknown_pairs,contacts,shape,assembly_id,new_cluster_pair,cluster1,cluster2,new_shape1,new_shape2,control_shape
0,0,10BL,1,10bl-assembly1.cif.gz,10bl-assembly1.npy,10BL_1,10BL_1,1,1,1,...,20133,139,361x361,10BL-1,"fix_18845_100,fix_18845_100",fix_18845_100,fix_18845_100,361,361,361x361
1,1,10FT,1,10ft-assembly1.cif.gz,10ft-assembly1.npy,10FT_1,10FT_2,1,2,1,...,706281,125,817x1011,10FT-1,"fix_14530_100,fix_37352_100",fix_14530_100,fix_37352_100,817,1011,817x1011
2,2,10JU,1,10ju-assembly1.cif.gz,10ju-assembly1.npy,10JU_1,10JU_2,1,2,1,...,63004,41,362x362,10JU-1,"fix_18848_100,fix_18846_100",fix_18848_100,fix_18846_100,362,362,362x362
3,3,10LI,1,10li-assembly1.cif.gz,10li-assembly1.npy,10LI_1,10LI_1,1,1,1,...,15040,262,478x478,10LI-1,"fix_31296_100,fix_31296_100",fix_31296_100,fix_31296_100,478,478,478x478
4,4,10NM,1,10nm-assembly1.cif.gz,10nm-assembly1.npy,10NM_1,10NM_1,1,1,1,...,44864,277,496x496,10NM-1,"fix_20451_100,fix_20451_100",fix_20451_100,fix_20451_100,496,496,496x496
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22088,22088,9ZMZ,1,9zmz-assembly1.cif.gz,9zmz-assembly1.npy,9ZMZ_1,9ZMZ_1,1,1,1,...,2743,203,112x112,9ZMZ-1,"fix_39175_100,fix_39175_100",fix_39175_100,fix_39175_100,112,112,112x112
22089,22089,9ZNN,1,9znn-assembly1.cif.gz,9znn-assembly1.npy,9ZNN_1,9ZNN_1,1,1,1,...,26953,117,419x419,9ZNN-1,"fix_32925_100,fix_32925_100",fix_32925_100,fix_32925_100,419,419,419x419
22090,22090,9ZNO,1,9zno-assembly1.cif.gz,9zno-assembly1.npy,9ZNO_1,9ZNO_1,1,1,1,...,23664,67,365x365,9ZNO-1,"fix_13069_100,fix_13069_100",fix_13069_100,fix_13069_100,365,365,365x365
22091,22091,9ZVM,1,9zvm-assembly1.cif.gz,9zvm-assembly1.npy,9ZVM_1,9ZVM_1,1,1,1,...,537999,62,860x860,9ZVM-1,"fix_24677_100,fix_24677_100",fix_24677_100,fix_24677_100,860,860,860x860


In [18]:
summary_df[["old_shape1", "old_shape2"]] = summary_df["shape"].str.split("x", expand=True)

summary_df

,row_index,pdb_id,assembly_number,structure_file,output_file,requested_entity_name_1,requested_entity_name_2,requested_entity_1,requested_entity_2,remapped_entity_1,...,shape,assembly_id,new_cluster_pair,cluster1,cluster2,new_shape1,new_shape2,control_shape,old_shape1,old_shape2
0,0,10BL,1,10bl-assembly1.cif.gz,10bl-assembly1.npy,10BL_1,10BL_1,1,1,1,...,361x361,10BL-1,"fix_18845_100,fix_18845_100",fix_18845_100,fix_18845_100,361,361,361x361,361,361
1,1,10FT,1,10ft-assembly1.cif.gz,10ft-assembly1.npy,10FT_1,10FT_2,1,2,1,...,817x1011,10FT-1,"fix_14530_100,fix_37352_100",fix_14530_100,fix_37352_100,817,1011,817x1011,817,1011
2,2,10JU,1,10ju-assembly1.cif.gz,10ju-assembly1.npy,10JU_1,10JU_2,1,2,1,...,362x362,10JU-1,"fix_18848_100,fix_18846_100",fix_18848_100,fix_18846_100,362,362,362x362,362,362
3,3,10LI,1,10li-assembly1.cif.gz,10li-assembly1.npy,10LI_1,10LI_1,1,1,1,...,478x478,10LI-1,"fix_31296_100,fix_31296_100",fix_31296_100,fix_31296_100,478,478,478x478,478,478
4,4,10NM,1,10nm-assembly1.cif.gz,10nm-assembly1.npy,10NM_1,10NM_1,1,1,1,...,496x496,10NM-1,"fix_20451_100,fix_20451_100",fix_20451_100,fix_20451_100,496,496,496x496,496,496
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22088,22088,9ZMZ,1,9zmz-assembly1.cif.gz,9zmz-assembly1.npy,9ZMZ_1,9ZMZ_1,1,1,1,...,112x112,9ZMZ-1,"fix_39175_100,fix_39175_100",fix_39175_100,fix_39175_100,112,112,112x112,112,112
22089,22089,9ZNN,1,9znn-assembly1.cif.gz,9znn-assembly1.npy,9ZNN_1,9ZNN_1,1,1,1,...,419x419,9ZNN-1,"fix_32925_100,fix_32925_100",fix_32925_100,fix_32925_100,419,419,419x419,419,419
22090,22090,9ZNO,1,9zno-assembly1.cif.gz,9zno-assembly1.npy,9ZNO_1,9ZNO_1,1,1,1,...,365x365,9ZNO-1,"fix_13069_100,fix_13069_100",fix_13069_100,fix_13069_100,365,365,365x365,365,365
22091,22091,9ZVM,1,9zvm-assembly1.cif.gz,9zvm-assembly1.npy,9ZVM_1,9ZVM_1,1,1,1,...,860x860,9ZVM-1,"fix_24677_100,fix_24677_100",fix_24677_100,fix_24677_100,860,860,860x860,860,860


In [19]:
# Check whether any shapes are not identical
# faulty_summary_df = summary_df[(summary_df["new_shape1"].eq(summary_df["old_shape1"]) | summary_df["new_shape1"].eq(summary_df["old_shape2"])) & (summary_df["new_shape2"].eq(summary_df["old_shape1"]) | summary_df["new_shape2"].eq(summary_df["old_shape2"]))].copy()
faulty_summary_df = summary_df[summary_df["shape"] != summary_df["control_shape"]]

faulty_summary_df

,row_index,pdb_id,assembly_number,structure_file,output_file,requested_entity_name_1,requested_entity_name_2,requested_entity_1,requested_entity_2,remapped_entity_1,...,shape,assembly_id,new_cluster_pair,cluster1,cluster2,new_shape1,new_shape2,control_shape,old_shape1,old_shape2


In [20]:
# are any faulty ones in the interaction DF ??
faulty_summary_df[faulty_summary_df["pdb_id"].isin(interaction_df["pdb_id"])]

,row_index,pdb_id,assembly_number,structure_file,output_file,requested_entity_name_1,requested_entity_name_2,requested_entity_1,requested_entity_2,remapped_entity_1,...,shape,assembly_id,new_cluster_pair,cluster1,cluster2,new_shape1,new_shape2,control_shape,old_shape1,old_shape2


## new approach

In [17]:
summary_df = pd.read_csv(SUMMARY_FILE, sep="\t")
entitiy_sequence_df = pd.read_csv(ENTITY_SEQUENCE_PATH, sep="\t")
interaction_df = pd.read_csv(INTERACTION_FILE, sep="\t")

summary_df = summary_df.merge(interaction_df[["pdb_id", "entity_pair"]], on="pdb_id", how="left")

summary_df.head()


,row_index,pdb_id,assembly_number,structure_file,output_file,requested_entity_name_1,requested_entity_name_2,requested_entity_1,requested_entity_2,remapped_entity_1,remapped_entity_2,chain_1,chain_2,residues_1,residues_2,contacts,shape,entity_pair
0,0,10BL,1,10bl-assembly1.cif.gz,10bl-assembly1.npy,10BL_1,10BL_1,1,1,1,1,A,B,338,326,139,338x326,"10BL_1,10BL_1"
1,1,10FT,1,10ft-assembly1.cif.gz,10ft-assembly1.npy,10FT_1,10FT_2,1,2,1,2,A,B,213,562,125,213x562,"10FT_1,10FT_2"
2,2,10JU,1,10ju-assembly1.cif.gz,10ju-assembly1.npy,10JU_1,10JU_2,1,2,1,2,A,B,270,252,41,270x252,"10JU_1,10JU_2"
3,3,10LI,1,10li-assembly1.cif.gz,10li-assembly1.npy,10LI_1,10LI_1,1,1,1,1,A,A-2,462,462,262,462x462,"10LI_1,10LI_1"
4,4,10NM,1,10nm-assembly1.cif.gz,10nm-assembly1.npy,10NM_1,10NM_1,1,1,1,1,A,B,448,449,277,448x449,"10NM_1,10NM_1"
